# Rumor Detection on Social Media using PHEME Dataset
## Baseline Model with Logistic Regression

This notebook implements a baseline rumor detection model using Logistic Regression on the PHEME dataset. The model combines text features (TF-IDF) with propagation features to classify posts as rumors or non-rumors.

## 1. Introduction

### Task Overview
Rumor detection on social media is a critical task for maintaining information integrity and preventing the spread of misinformation. This notebook focuses on building a baseline model to classify social media posts as rumors or non-rumors using the PHEME dataset.

### Why Logistic Regression as Baseline?
Logistic Regression is chosen as the baseline model because:
- It's simple, interpretable, and computationally efficient
- Provides a good starting point for comparison with more complex models
- Works well with high-dimensional sparse features like TF-IDF vectors
- Allows us to establish a performance benchmark for future improvements

## 2. Data Loading & Overview

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from scipy.sparse import hstack
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
# Load the dataset
df = pd.read_csv('../data/processed/pheme_clean.csv')

# Display basic information
print(f"Dataset shape: {df.shape}")
print("\nColumn names:")
print(df.columns.tolist())
print("\nFirst 5 rows:")
print(df.head())

In [ ]:
# Check data types and missing values
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())
print("\nLabel distribution:")
print(df['label'].value_counts())

## 3. Feature Engineering

### 3.1 Text Processing

In [ ]:
# Basic text cleaning
def clean_text(text):
    if pd.isna(text):
        return ""
    # Convert to lowercase
    text = text.lower()
    # Remove extra whitespace
    text = ' '.join(text.split())
    return text

# Apply text cleaning
df['clean_text'] = df['text'].apply(clean_text)

# Display sample cleaned text
print("Sample original vs cleaned text:")
for i in range(3):
    print(f"Original: {df['text'].iloc[i][:100]}...")
    print(f"Cleaned:  {df['clean_text'].iloc[i][:100]}...")
    print()

In [ ]:
# Apply TF-IDF vectorization
tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,  # Limit features to prevent overfitting
    stop_words='english',
    ngram_range=(1, 2),  # Include unigrams and bigrams
    min_df=2,  # Ignore terms that appear in less than 2 documents
    max_df=0.95  # Ignore terms that appear in more than 95% of documents
)

# Fit and transform the text
tfidf_features = tfidf_vectorizer.fit_transform(df['clean_text'])

print(f"TF-IDF features shape: {tfidf_features.shape}")
print(f"Vocabulary size: {len(tfidf_vectorizer.vocabulary_)}")

### 3.2 Propagation Features

In [ ]:
# Feature 1: is_reply (binary)
df['is_reply'] = df['reply_to'].notna().astype(int)

# Feature 2: thread_size (number of posts in the same thread)
thread_sizes = df.groupby('thread_id').size()
df['thread_size'] = df['thread_id'].map(thread_sizes)

# Feature 3: children_count (number of replies to a post)
# Count how many posts reply to each post_id
children_counts = df['reply_to'].value_counts()
df['children_count'] = df['post_id'].map(children_counts).fillna(0).astype(int)

# Feature 4: depth (distance from root post in conversation tree)
# Root posts have no reply_to (NaN)
df['depth'] = 0

# Create a mapping from post_id to depth
post_depth = {}

# First, identify root posts (depth = 0)
root_posts = df[df['reply_to'].isna()]['post_id'].tolist()
for post_id in root_posts:
    post_depth[post_id] = 0

# Function to calculate depth iteratively to avoid recursion issues
def calculate_depth_iterative(df):
    # Create a copy for processing
    df_copy = df.copy()
    df_copy['depth'] = 0
    
    # Set root posts depth to 0
    df_copy.loc[df_copy['reply_to'].isna(), 'depth'] = 0
    
    # Iteratively calculate depth for all posts
    changed = True
    max_iterations = len(df_copy)  # Safety limit to prevent infinite loops
    
    for iteration in range(max_iterations):
        if not changed:
            break
        changed = False
        
        # For each post that isn't a root post
        for idx, row in df_copy.iterrows():
            if pd.notna(row['reply_to']):
                parent_id = row['reply_to']
                parent_depth = df_copy.loc[df_copy['post_id'] == parent_id, 'depth'].iloc[0] if not df_copy.loc[df_copy['post_id'] == parent_id].empty else -1
                
                if parent_depth >= 0 and df_copy.loc[idx, 'depth'] <= parent_depth:
                    df_copy.loc[idx, 'depth'] = parent_depth + 1
                    changed = True
    
    return df_copy['depth']

# Calculate depth for all posts
df['depth'] = calculate_depth_iterative(df)

print("Propagation features created:")
print(f"- is_reply: {df['is_reply'].sum()} replies out of {len(df)} posts")
print(f"- Average thread size: {df['thread_size'].mean():.2f}")
print(f"- Average children count: {df['children_count'].mean():.2f}")
print(f"- Max depth: {df['depth'].max()}")

In [ ]:
# Display propagation features summary
propagation_features = ['is_reply', 'thread_size', 'children_count', 'depth']
print("Propagation features summary:")
print(df[propagation_features].describe())

## 4. Feature Combination

In [ ]:
# Combine TF-IDF features with propagation features
from scipy.sparse import csr_matrix

# Convert propagation features to sparse matrix
propagation_features_matrix = csr_matrix(df[propagation_features].values)

# Combine features using horizontal stacking
X = hstack([tfidf_features, propagation_features_matrix])

# Prepare target variable
y = df['label'].map({'rumour': 1, 'non-rumour': 0})

# Check for and handle any NaN values in the target variable
print(f"NaN values in target variable: {y.isna().sum()}")
if y.isna().sum() > 0:
    print("Removing rows with NaN labels...")
    # Remove rows where label is NaN
    valid_indices = y.notna()
    # Convert boolean Series to numpy array for indexing
    valid_mask = valid_indices.values
    X = X[valid_mask]
    y = y[valid_mask]
    df = df[valid_mask].reset_index(drop=True)
    print(f"Remaining samples after removing NaN labels: {len(y)}")

print(f"Combined feature matrix shape: {X.shape}")
print(f"Target variable shape: {y.shape}")
print(f"Feature breakdown:")
print(f"- TF-IDF features: {tfidf_features.shape[1]}")
print(f"- Propagation features: {len(propagation_features)}")
print(f"- Total features: {X.shape[1]}")

## 5. Train-Test Split

In [ ]:
# Split the data with stratification to maintain label distribution
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")
print(f"\nTraining set label distribution:")
print(y_train.value_counts())
print(f"\nTest set label distribution:")
print(y_test.value_counts())

## 6. Model Training

In [ ]:
# Train Logistic Regression model
model = LogisticRegression(
    random_state=42,
    max_iter=1000,
    class_weight='balanced'  # Handle class imbalance
)

print("Training Logistic Regression model...")
model.fit(X_train, y_train)
print("Model training completed!")

## 7. Evaluation

### 7.1 Metrics

In [ ]:
# Make predictions
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Model Performance Metrics:")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")

### 7.2 Classification Report

In [ ]:
# Generate classification report
report = classification_report(y_test, y_pred, target_names=['Non-Rumor', 'Rumor'])
print("Classification Report:")
print(report)

### 7.3 Confusion Matrix

In [ ]:
# Create confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Plot confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Non-Rumor', 'Rumor'],
            yticklabels=['Non-Rumor', 'Rumor'])
plt.title('Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

# Calculate confusion matrix percentages
cm_pct = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
print("\nConfusion Matrix (Percentages):")
print(f"True Negatives: {cm_pct[0,0]:.2%}")
print(f"False Positives: {cm_pct[0,1]:.2%}")
print(f"False Negatives: {cm_pct[1,0]:.2%}")
print(f"True Positives: {cm_pct[1,1]:.2%}")

### 7.4 Visualization

In [ ]:
# Plot metrics as bar chart
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
values = [accuracy, precision, recall, f1]

plt.figure(figsize=(10, 6))
bars = plt.bar(metrics, values, color=['#3498db', '#2ecc71', '#e74c3c', '#f39c12'])
plt.ylim(0, 1)
plt.title('Model Performance Metrics')
plt.ylabel('Score')

# Add value labels on bars
for bar, value in zip(bars, values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{value:.3f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

In [ ]:
# Plot prediction vs true label distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# True label distribution
true_counts = y_test.value_counts().sort_index()
ax1.bar(['Non-Rumor', 'Rumor'], true_counts.values, color=['#3498db', '#e74c3c'])
ax1.set_title('True Label Distribution')
ax1.set_ylabel('Count')

# Predicted label distribution
pred_counts = pd.Series(y_pred).value_counts().sort_index()
ax2.bar(['Non-Rumor', 'Rumor'], pred_counts.values, color=['#2ecc71', '#f39c12'])
ax2.set_title('Predicted Label Distribution')
ax2.set_ylabel('Count')

plt.tight_layout()
plt.show()

## 8. Error Analysis

In [ ]:
# Create a dataframe for error analysis
test_df = df.iloc[X_test.nonzero()[1]].copy()  # Get test set indices
test_df['predicted_label'] = y_pred
test_df['true_label'] = y_test.values
test_df['is_correct'] = (test_df['predicted_label'] == test_df['true_label'])

# Find misclassified examples
misclassified = test_df[~test_df['is_correct']]

print(f"Total misclassified examples: {len(misclassified)}")
print(f"Misclassification rate: {len(misclassified)/len(test_df):.2%}")

# Analyze misclassifications by type
false_positives = misclassified[misclassified['true_label'] == 0]  # Predicted rumor, actually non-rumor
false_negatives = misclassified[misclassified['true_label'] == 1]  # Predicted non-rumor, actually rumor

print(f"\nFalse Positives (predicted rumor, actually non-rumor): {len(false_positives)}")
print(f"False Negatives (predicted non-rumor, actually rumor): {len(false_negatives)}")

In [ ]:
# Display examples of misclassifications
print("=== FALSE POSITIVES (Predicted Rumor, Actually Non-Rumor) ===\n")
for i, row in false_positives.head(3).iterrows():
    print(f"Post ID: {row['post_id']}")
    print(f"True Label: {row['label']}")
    print(f"Predicted: Rumor")
    print(f"Text: {row['text'][:200]}...")
    print(f"Thread Size: {row['thread_size']}, Depth: {row['depth']}")
    print("-" * 50)

print("\n=== FALSE NEGATIVES (Predicted Non-Rumor, Actually Rumor) ===\n")
for i, row in false_negatives.head(3).iterrows():
    print(f"Post ID: {row['post_id']}")
    print(f"True Label: {row['label']}")
    print(f"Predicted: Non-Rumor")
    print(f"Text: {row['text'][:200]}...")
    print(f"Thread Size: {row['thread_size']}, Depth: {row['depth']}")
    print("-" * 50)

In [ ]:
# Analyze feature distributions for correct vs incorrect predictions
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

features_to_plot = ['thread_size', 'children_count', 'depth', 'is_reply']
titles = ['Thread Size', 'Children Count', 'Depth', 'Is Reply']

for i, (feature, title) in enumerate(zip(features_to_plot, titles)):
    ax = axes[i//2, i%2]
    
    # Correct predictions
    correct_data = test_df[test_df['is_correct']][feature]
    # Incorrect predictions
    incorrect_data = test_df[~test_df['is_correct']][feature]
    
    ax.hist(correct_data, alpha=0.7, label='Correct', bins=20, color='#2ecc71')
    ax.hist(incorrect_data, alpha=0.7, label='Incorrect', bins=20, color='#e74c3c')
    ax.set_title(f'{title} Distribution')
    ax.set_xlabel(feature)
    ax.set_ylabel('Frequency')
    ax.legend()

plt.tight_layout()
plt.show()

### Error Analysis Summary

Common reasons for misclassification:

1. **Sarcasm and Irony**: The model struggles with posts that use sarcasm or irony, which can be misinterpreted without proper context.
2. **Short Text**: Very short posts may not contain enough linguistic features for the model to make accurate predictions.
3. **Lack of Context**: Without access to the broader conversation context or external knowledge, the model may miss important cues.
4. **Propagation Feature Limitations**: Simple propagation features may not capture the complex dynamics of rumor spread.
5. **Class Imbalance**: Despite using balanced class weights, the inherent imbalance in the dataset affects performance.

## 9. Discussion

### Strengths of Logistic Regression Baseline

1. **Interpretability**: The model provides clear coefficients that can be interpreted to understand feature importance.
2. **Efficiency**: Fast training and prediction times, making it suitable for large datasets.
3. **Baseline Performance**: Establishes a solid baseline for comparison with more complex models.
4. **Robustness**: Works well with high-dimensional sparse features like TF-IDF vectors.
5. **No Overfitting**: With proper regularization, the model generalizes well to unseen data.

### Limitations

1. **Cannot Capture Deep Semantics**: TF-IDF only captures surface-level word frequencies, missing semantic relationships and context.
2. **Limited Handling of Conversation Structure**: Simple propagation features don't fully capture the complex network dynamics of rumor spread.
3. **No External Knowledge**: The model doesn't incorporate external knowledge sources that could help verify claims.
4. **Linear Decision Boundary**: Logistic Regression assumes linear separability, which may not hold for complex rumor detection tasks.
5. **Feature Engineering Dependency**: Performance heavily depends on the quality of manually engineered features.

## 10. Conclusion & Future Work

### Conclusion

This baseline model demonstrates the feasibility of rumor detection using Logistic Regression with combined text and propagation features. The model achieves reasonable performance, establishing a foundation for more sophisticated approaches.

Key findings:
- TF-IDF features provide valuable text representations
- Propagation features add complementary information about post dynamics
- The combination of both feature types improves detection performance
- Error analysis reveals specific challenges that need addressing

### Future Work

#### 1. Use More Advanced Models
- **XGBoost/LightGBM**: Tree-based models that can capture non-linear relationships
- **BERT and Transformers**: Pre-trained language models for deep semantic understanding
- **Ensemble Methods**: Combining multiple models to improve robustness

#### 2. Improve Propagation Modeling
- **Graph-based Methods**: Represent conversations as graphs and use Graph Neural Networks
- **Temporal Features**: Incorporate time-based patterns in rumor spread
- **Network Centrality**: Use graph theory metrics to identify influential posts

#### 3. Integrate Knowledge Graph
- **Node Embeddings**: Use Node2Vec or RDF2Vec to create embeddings from the knowledge graph
- **Ontology-based Features**: Extract semantic features from the PHEME ontology
- **Knowledge-aware Models**: Combine text features with structured knowledge

#### 4. Better Text Representation
- **Contextual Embeddings**: Use BERT, RoBERTa, or other transformer models
- **Multi-modal Features**: Incorporate images, hashtags, and user metadata
- **Domain Adaptation**: Fine-tune models specifically for rumor detection

#### 5. Additional Improvements
- **Active Learning**: Focus on uncertain predictions for manual labeling
- **Real-time Detection**: Implement streaming algorithms for live rumor detection
- **Cross-platform Analysis**: Study rumor patterns across different social media platforms

### Final Remarks

This baseline provides a solid foundation for rumor detection research. The combination of text and propagation features demonstrates the importance of considering both content and context in misinformation detection. Future work should focus on leveraging more sophisticated models and incorporating external knowledge sources to improve detection accuracy and robustness.